# John Deere Agricultural Object Detection
## Notebook 02: Model Evaluation and In-Depth Error Analysis

**Objective:** Evaluate trained YOLO11 models on agricultural validation data, quantify performance metrics, and perform rigorous error analysis (False Positives, False Negatives, scale variations, and occlusion).

---
### Why Detailed Evaluation & Error Analysis Matters
In deep learning and machine perception, a single summary metric (e.g. mAP@50) never tells the full engineering story.
For agricultural applications like John Deere autonomous equipment and operator assistance:
- **Precision** indicates how many detected objects are actually real (avoiding false emergency stops).
- **Recall** indicates how many real obstacles/workers were detected (avoiding dangerous collisions).
- **Error Analysis** isolates exactly *why* and *when* the model fails (e.g., small distant objects, crop occlusion, lighting glare).

> **Academic Integrity Principle:** Metrics are only computed on actual trained weights. If weights are not trained, the notebook demonstrates the exact mathematical formulas and error analysis pipeline.

In [ ]:
import sys
import json
from pathlib import Path

# Set project root path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

from src.utils import load_config, compute_iou, xywh_to_xyxy
from src.evaluate import check_model_availability, run_error_analysis

config = load_config(PROJECT_ROOT / "configs" / "config.yaml")
class_names = config["dataset"]["classes"]
model_path = PROJECT_ROOT / "models" / config["model"]["best_model_name"]

print("Model checkpoint path:", model_path)
print("Weights available:    ", check_model_availability(model_path))
print("Target Classes:       ", class_names)

### 1. Object Detection Metric Foundations
In object detection, a prediction $[\hat{x}_1, \hat{y}_1, \hat{x}_2, \hat{y}_2]$ is evaluated against ground truth $[x_1, y_1, x_2, y_2]$ using:

$$\text{IoU} = \frac{\text{Area of Overlap}}{\text{Area of Union}} = \frac{B_{\text{pred}} \cap B_{\text{gt}}}{B_{\text{pred}} \cup B_{\text{gt}}}$$

- **True Positive (TP):** $\text{IoU} \ge 0.5$ with correct class.
- **False Positive (FP):** $\text{IoU} < 0.5$ (misplaced box, duplicate box, or hallucinated background object).
- **False Negative (FN):** Ground truth object not detected at $\text{IoU} \ge 0.5$.

$$\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}, \quad \text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}}, \quad \text{F1} = \frac{2 \cdot P \cdot R}{P + R}$$

Let's demonstrate IoU computation interactively:

In [ ]:
# Interactive IoU demonstration
gt_box = [50.0, 50.0, 200.0, 250.0]       # Ground truth: tractor
pred_good = [60.0, 55.0, 205.0, 245.0]   # High overlap prediction
pred_poor = [150.0, 180.0, 300.0, 320.0] # Poor overlap prediction

iou_good = compute_iou(gt_box, pred_good)
iou_poor = compute_iou(gt_box, pred_poor)

print(f"IoU (High overlap): {iou_good:.4f} -> {'True Positive (IoU >= 0.5)' if iou_good >= 0.5 else 'False Positive'}")
print(f"IoU (Poor overlap): {iou_poor:.4f} -> {'True Positive (IoU >= 0.5)' if iou_poor >= 0.5 else 'False Positive'}")

# Visualize boxes
canvas = Image.new("RGB", (350, 350), color=(240, 240, 240))
draw = ImageDraw.Draw(canvas)
draw.rectangle(gt_box, outline=(0, 150, 0), width=3)           # Green = Ground Truth
draw.rectangle(pred_good, outline=(0, 100, 255), width=2)       # Blue = Good Pred
draw.rectangle(pred_poor, outline=(255, 0, 0), width=2)         # Red = Poor Pred

plt.figure(figsize=(5, 5))
plt.imshow(canvas)
plt.title("IoU Comparison: Ground Truth (Green), Good (Blue), Poor (Red)")
plt.axis("off")
plt.show()

### 2. Loading Trained Model Evaluation Metrics
If the model has been trained, we load the structured metrics exported to `outputs/metrics/evaluation_results.json`.

In [ ]:
metrics_file = PROJECT_ROOT / "outputs" / "metrics" / "evaluation_results.json"
if metrics_file.is_file():
    with open(metrics_file, "r") as f:
        metrics = json.load(f)
    print("Loaded Trained Metrics:")
    print(json.dumps(metrics, indent=2))
else:
    print("=" * 75)
    print("Model training has not yet been executed; metrics are not available.")
    print("To train the model and generate real evaluation metrics, run:")
    print("    python src/train.py --epochs 25 --batch-size 16")
    print("=" * 75)

### 3. Error Analysis Framework
We now run the automated Error Analysis pipeline to break down:
1. **Scale Sensitivity:** Are small objects (e.g. distant workers) missed more often than large machinery?
2. **False Alarm Rate:** What causes false positives in agricultural fields?

In [ ]:
# Demonstrate error analysis logic with simulated validation predictions
simulated_preds = [
    {"image_id": "val_01", "box": [100.0, 100.0, 300.0, 300.0], "class_id": 0, "score": 0.92},
    {"image_id": "val_01", "box": [400.0, 200.0, 430.0, 310.0], "class_id": 1, "score": 0.81},
    {"image_id": "val_02", "box": [150.0, 150.0, 250.0, 250.0], "class_id": 0, "score": 0.40}, # Low conf FP
]

simulated_gts = [
    {"image_id": "val_01", "box": [95.0, 105.0, 295.0, 305.0], "class_id": 0, "area": 0.25},  # Matched tractor
    {"image_id": "val_01", "box": [398.0, 198.0, 432.0, 312.0], "class_id": 1, "area": 0.03},  # Matched person
    {"image_id": "val_02", "box": [500.0, 210.0, 520.0, 250.0], "class_id": 1, "area": 0.01},  # Missed distant person (FN)
]

error_report = run_error_analysis(simulated_preds, simulated_gts, iou_threshold=0.5)
print("Error Analysis Results:")
print(json.dumps(error_report, indent=2))

### 4. Confidence Threshold Ablation: Precision vs. Recall Tradeoff
In agricultural robotics and autonomous tractors, setting the confidence threshold $\tau$ involves a fundamental trade-off:
- **High Threshold (e.g. $\tau = 0.7$):** Precision is high (few false alarms), but Recall drops (small or occluded workers might be missed).
- **Low Threshold (e.g. $\tau = 0.2$):** Recall is high (safe: catches almost all obstacles), but Precision drops (frequent false alarms on field clutter).

Let's plot how Precision and Recall vary across thresholds $\tau \in [0.1, 0.9]$:

In [ ]:
thresholds = np.linspace(0.1, 0.9, 9)
# Illustrative precision-recall curves showing the characteristic trade-off
precision_curve = [0.55, 0.65, 0.74, 0.81, 0.88, 0.92, 0.95, 0.97, 0.98]
recall_curve =    [0.96, 0.93, 0.89, 0.84, 0.78, 0.70, 0.58, 0.44, 0.28]
f1_curve = [2 * p * r / (p + r) for p, r in zip(precision_curve, recall_curve)]

plt.figure(figsize=(8, 4.5))
plt.plot(thresholds, precision_curve, marker="o", label="Precision (Safety vs False Alarms)", color="#1f77b4", linewidth=2)
plt.plot(thresholds, recall_curve, marker="s", label="Recall (Obstacle Detection Rate)", color="#2ca02c", linewidth=2)
plt.plot(thresholds, f1_curve, marker="^", label="F1 Score (Harmonic Balance)", color="#d62728", linestyle="--", linewidth=2)

best_idx = np.argmax(f1_curve)
plt.axvline(thresholds[best_idx], color="gray", linestyle=":", label=f"Optimal F1 Threshold ({thresholds[best_idx]:.2f})")

plt.title("Confidence Threshold Tuning for Agricultural Object Detection", fontsize=13, pad=10)
plt.xlabel("Confidence Threshold (\tau)", fontsize=11)
plt.ylabel("Metric Value", fontsize=11)
plt.ylim(0, 1.05)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()

### 5. Detailed Qualitative Error Analysis Summary

| Error Mode | Root Cause in Agriculture | Mitigation Strategy |
| :--- | :--- | :--- |
| **Small Object False Negatives** | Distant farm workers subtend only 15–30 pixels in wide field shots. Feature downsampling loses localized gradients. | Increase input resolution from 640 to 1280 (`imgsz=1280`), or use mosaic augmentation. |
| **Occlusion False Negatives** | Field workers crouching behind crops, or tractors partially occluded behind grain wagons. | Augment with Cutout / Mixup; leverage temporal multi-frame fusion. |
| **Background False Positives** | Dark soil mounds, irrigation pipes, or equipment shadows hallucinated as machinery. | Expand background image collection (images with no objects) during training. |
| **Lighting Glare** | Direct morning/evening sun flare washes out camera sensors in outdoor operations. | HSV brightness and saturation augmentation (`hsv_s: 0.7`, `hsv_v: 0.4`). |